<a href="https://colab.research.google.com/github/claudinha08/My_course_SCTECH/blob/main/Desafio_Extra_SCTECH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy matplotlib seaborn plotly scikit-learn

# Importanto bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.preprocessing import RobustScaler
sns.set_style('whitegrid')

In [ ]:
import plotly.express as px


#Carregando dados (importados para drive)

In [ ]:
#carregar o drive
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/datasetes/Sample_Superstore.csv'

In [ ]:
path = '/content/drive/MyDrive/datasetes/Sample_Superstore.csv'
df = pd.read_csv(path, encoding='latin-1')

print(df.head())

In [ ]:
df.shape

In [ ]:
#tipos
df.info()
df.describe()


**Número de linha nulas: 0**

**Memória do data frame: 1.6+ MB**

In [ ]:
print(df.dtypes)
#é possivel ver as colunas

# Verificar dados duplicados

In [ ]:
duplicateRows = df[df.duplicated()]
print(f"número de dados duplicados: {len(duplicateRows)}")

#remover valores nulos
df.dropna(subset=[c for c in ['sales', 'profit'] if c in df.columns], inplace=True)

#converter datas
if 'Order Date' in df.columns:
   df['order_date'] = pd.to_datetime(df['Order Date'], dayfirst = False)

# converter números (vendas, lucro, desconto, quantidade)
for col in ['Sales', 'Profit', 'Discount', 'Quantitu']:
  if col in df.columns:
    df[col] = pd.to_numeric(df[col])



In [ ]:
#padronizar nome das colunas
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print(df.columns)


# Detecção de Outlier

In [ ]:
#inspecionar distribuição
sns.boxplot(x=df['sales'])
plt.show()


In [ ]:
#limitar extremos - tratar outliers
def winsorize_series(s, lower=0.01, upper=0.99):
  lo = s.quantile(lower)
  hi = s.quantile(upper)
  return s.clip(lo,hi)

df['sales_wins'] = winsorize_series(df['sales'])
df['profit_wins'] = winsorize_series(df['profit'])

sns.boxplot(x=df['sales'])
plt.show()
sns.boxplot(x=df['profit'])
plt.show()

In [ ]:
# Métricas derivadas
df['margin'] = df['profit'] / df['sales']

# Filtros, ordenação e agrupamento

In [ ]:
#filtro
print(df[df['profit'] >= 28.65].head()) #valores de lucro maiores que a média


In [ ]:
#agrupamento (GroupBy)
#vendas e lucro por categoria
vendas_lucro = df.groupby('category').agg(
    total_sales = ('sales','sum'),
    total_profit = ('profit','sum'),
    count_orders = ('order_id','nunique') if 'order_id' in df.columns else ('order_id', 'count')
    ).reset_index().sort_values('total_sales', ascending=False)

print(vendas_lucro)

# Visualizações (matplotlib/seaborn)

In [ ]:
#serie temporal (matplotlib)
plt.figure(figsize=(12,5))
df = df.loc[:, ~df.columns.duplicated(keep='last')] #feito com dicas, rever
monthly = df.set_index('order_date').resample('ME').sum()[['sales','profit']]
monthly['sales'].plot(label='Sales')
monthly['profit'].plot(label='Profit')
plt.legend(); plt.title('Vendas e Lucro'); plt.show()

# Barra: vendas por categoria (seaborn)
plt.figure(figsize=(8,5))
sns.barplot(data=vendas_lucro, x='category', y='total_sales')
plt.title('Total de Vendas por Categoria'); plt.xticks(rotation=30); plt.show()

# Análises Vendas, Lucro e desconto

In [ ]:
df[['sales','profit']].mean()
display(df[['sales','profit']].mean())

#Análise do desconto por região
region_profit = df.groupby('region')['profit'].sum().sort_values(ascending=False)
print(region_profit)

region_profit.plot(kind='bar', color='green')
plt.title('Total de desconto por região')
plt.ylabel('Profit')
plt.xticks(rotation=0)
plt.show()

#análise da venda por região
region_sales = df.groupby('region')['sales'].sum().sort_values(ascending=False)
print(region_sales)

region_sales.plot(kind='bar', color='green')
plt.title('Total de venda por região')
plt.ylabel('Sales')
plt.xticks(rotation=0)
plt.show()

